In [4]:
import numpy as np
import matplotlib.pyplot as plt
from Scripts.drone_mass import DroneMass
from Scripts.drone_battery import Battery
from Scripts.drone_geometry import DroneGeometry
from Scripts.beams import BeamType
from Scripts.mission_profile import MissionProfile
from Scripts.drone_power import DronePower
from Scripts.drone_power import BatterySystem
from Scripts.drone_forces import DroneForces
from Scripts.drone_fea import DroneFEA
import Scripts.unit_conversions as uc

In [5]:
drag_coeff = 0.256

# Battery geometry
battery_1 = Battery(4.292e-3, np.array([0, 86.75, -50]), np.array([260.5, 123.5, 63.5]))
battery_2 = Battery(4.292e-3, np.array([0, -86.75, -50]), np.array([260.5, 123.5, 63.5]))

# Beam properties
arm_beam_properties = BeamType("annulus", [29, 25], "Aluminum7075-T6", "arm_beam")
strut_beam_properties = BeamType("annulus", [14, 12], "Aluminum7075-T6", "strut_beam")

# Drone geometry
drone_geometry = DroneGeometry("drone_test", 1000, 660, 8, arm_beam_properties, strut_beam_properties, [battery_1, battery_2])

# Mass analysis
drone_mass = DroneMass(drone_geometry)

total_mass = drone_mass.total_mass + (45 * 10 ** -3)
surface_area = drone_geometry.projected_surface_area

# Mission profile generation
mission_profile = MissionProfile("m1", 1, 0, 0)
mission_profile.add_segment("takeoff",2,0,2,0,1)
mission_profile.add_segment("climb",15,550000,100000,300,0)
mission_profile.add_segment("cruise",50,2000000,110000,300,0)
mission_profile.add_segment("descent",10,2200000,50000,0,0)
mission_profile.add_segment("loiter",5,2200000,50000,0,0)
mission_profile.add_segment("land",10,2400000,0,0,0)

# Drone force analysis
d1_forces = DroneForces(total_mass, surface_area, drag_coeff, mission_profile)

# Drone
battery_sys = BatterySystem("Tattu 40000mAh 6S 10C 22.8V High Voltage UAV Lipo Battery Pack with AS150+AS150", 22.8, 40000, 2)
drone_power = DronePower(8, battery_sys.milliwatt_hours, d1_forces)

drone_power.energy_consumption_calc(mission_profile.t_values, 2, 100, 1e-4)
drone_power.throttle_ratio_calc(16) # Power for max throttle in kw w/ full battery

d1_fea = DroneFEA(drone_geometry)

d1_fea.create_drone_slice_nodes()           # nominal radius | strut rad | number of blades
d1_fea.create_drone_slice_beams() # arm beam | strut beam
d1_fea.boundary_conditions_slice()
d1_fea.beam_system.add_force(np.array([0,0,-100]),"outer_node")
d1_fea.solve_fea()

Steps taken:  800
Max error:  1.4497305219629553e-05
[660.   0.   0.]
